# Cell Cycle Analysis

Template for standard cell cycle analysis with DNA/EdU/H3P channels.

User templates go in ~/.cellview/templates/. Any .ipynb file placed there will be discovered by list_templates() and will override a built-in template with the same name.                                                                                           
                                                            
So if a user creates ~/.cellview/templates/myscreen.ipynb, they can use it with:  
cellview explore 12345 --template myscreen            

In [ ]:
import os
os.environ["ENV"] = "production"

from cellview import cellview_load_data
# --- CellView Config ---
PLATE_IDS: list[int] = []  # will be filled by cellview explore

df, variable_names = cellview_load_data(*PLATE_IDS)

print(f"Loaded {len(df)} cells")
print(f"Cell lines: {df.cell_line.unique()}")
print(f"Variables: {variable_names}")
for variable_name in variable_names:
    print(f"{variable_name}: {df[variable_name].unique().tolist()}")

## Configure

Edit the experimental variables to match your expected plotting logic.
Your data will have experimental variables as separate columns (e.g. cell_line, sirna, drug, concentration). You need to combine these into a single condition column for plotting and define an order of categories in a list called `conditions`.

1) **Simple case** — one variable is already your condition (e.g. "siRNA"):
Check the order of the categories using
```python
df["siRNA"].unique()
```
If the order is correct you can assign this directly to the conditions list
```python
conditions = df["siRNA"].unique().tolist()
```
If not, create the conditions list manually by reordering the individual categories, or
use a map function with the ordering logic you want to apply.

2) **Two variables** — string concatenation:
```python
df["condition"] = df["drugA"] + " " + df["drugB"]
print(df["condition"].unique())
```
Then apply the same ordering logic as described above.

3) **More complex scenarios** —
You can use `np.where()`, `np.select()`, or other approaches to generate the correct categories for plotting. Ultimately you want to generate an ordered list `conditions` with the individual categories in the correct sequence.
```python
import numpy as np

drug_label = np.where(df["drug_treated"] == "1.0", "Drug", "DMSO")
conc_label = np.select(
    [df["high_dose"] == "1.0", df["low_dose"] == "1.0"],
    ["10\u00b5M", "1\u00b5M"],
    default="0\u00b5M",
)
df["condition"] = df["cell_line"] + ", " + drug_label + " " + conc_label
```

**Ordering conditions for plots:**
```python
conditions = ["ctrl", "drug_low", "drug_high"]
```

In [ ]:
# Set up your conditions here
# conditions = df["siRNA"].unique().tolist()

# Set cell_number for sampling in scatter/combo plots
cell_number = 3000

## Cell Cycle Combined Plot

To inspect the data and get a good overview of the cell cycle, use this function first.
It provides DNA histograms, EdU vs DNA scatter, and a stacked bar chart.

In [ ]:
from omero_screen_plots import combplot_cellcycle

for cell_line in df["cell_line"].unique():
    fig, axes = combplot_cellcycle(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        title=None,
        cell_number=cell_number,
        cc_phases=True,
        show_error_bars=True,
        fig_size=(14, 7),
        size_units="cm",
        save=False,
        path=None,
        file_format="png",
        dpi=300,
    )

## Feature Scatter Plot

Select features to plot as single-cell data. This gives you a good idea of distributions
and data quality. Use `df.columns` to check available feature columns.

In [ ]:
# df.columns  # uncomment to check available columns

In [ ]:
from omero_screen_plots import scatter_plot

for cell_line in df["cell_line"].unique():
    fig, axes = scatter_plot(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Plot features
        x_feature="integrated_int_DAPI_norm",
        y_feature="intensity_mean_EdU_nucleus_norm",
        cell_number=cell_number,
        # Hue settings
        hue=None,               # Auto-detects cell_cycle
        hue_order=None,
        palette=None,
        # Scale settings
        x_scale=None,            # Auto: log for DNA
        x_scale_base=2,
        y_scale=None,            # Auto: log for EdU, linear otherwise
        y_scale_base=2,
        # Axis limits & ticks
        x_limits=None,           # Auto-set for DNA content
        y_limits=None,
        x_ticks=None,
        y_ticks=None,
        # Scatter settings
        size=2,
        alpha=1.0,
        # KDE overlay
        kde_overlay=None,        # Auto for DNA vs EdU
        kde_cmap="rocket_r",
        kde_alpha=0.1,
        # Reference lines
        vline=None,              # Auto for DNA
        hline=None,              # Auto for EdU
        line_style="--",
        line_color="black",
        # Display
        grid=False,
        show_title=False,
        title=None,
        x_label=None,
        y_label=None,
        show_legend=False,
        legend_loc="best",
        legend_title=None,
        # Threshold
        threshold=None,
        # Figure
        fig_size=None,           # Auto: 7x7cm per condition
        size_units="cm",
        dpi=300,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        tight_layout=False,
        # Axes
        axes=None,
    )

## Combplot with Feature
Alternatively you might want to combine the single-cell cell cycle and feature analysis.
Use the `combplot_feature` function which generates DAPI histograms, DAPI/EdU scatter plots,
and feature scatter plots of your choice.

In [ ]:
from omero_screen_plots import combplot_feature                                                                                                                                                                                                                      
                 
for cell_line in df["cell_line"].unique():                                                                                                                                                                                                                           
    fig, axes = combplot_feature(                                                                                                                                                                                                                                    
        df=df,                                                                                                                                                                                                                                                       
        conditions=conditions,                                                                                                                                                                                                                                       
        feature="intensity_mean_p21_nucleus",                                                                                                                                                                                                                        
        threshold=5000,                                                                                                                                                                                                                                              
        condition_col="condition",                                                                                                                                                                                                                                   
        selector_col="cell_line",                                                                                                                                                                                                                                    
        selector_val=cell_line,                                                                                                                                                                                                                                      
        title=None,                                                                                                                                                                                                                                                  
        cell_number=3000,                                                                                                                                                                                                                                            
        fig_size=(10, 7),                                                                                                                                                                                                                                            
        size_units="cm",                                                                                                                                                                                                                                             
        save=False,                                       
        path=None,                                                                                                                                                                                                                                                   
        file_format="png",                                
        dpi=300,                                                                                                                                                                                                                                                     
    )                  

## Cell Counts

In [ ]:
from omero_screen_plots import count_plot
from omero_screen_plots.countplot_factory import PlotType

for cell_line in df["cell_line"].unique():
    fig, ax = count_plot(
        df=df,
        norm_control="ctrl",     # Set to your control condition for normalisation
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Display
        plot_type=PlotType.NORMALISED,  # or PlotType.ABSOLUTE
        title=None,
        # Grouping & layout
        group_size=1,
        within_group_spacing=0.2,
        between_group_gap=0.5,
        x_label=True,
        # Figure
        fig_size=(7, 7),
        size_units="cm",
        dpi=300,
        tight_layout=False,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        # Axes
        axes=None,
    )

## Cell Cycle Plots

There are two options to display the cell cycle distributions.
You can show each phase as a separate barplot with `cellcycle_plot`,
or stacked barplots with `cellcycle_stacked`.
With the stacked plots you have the option to show combined data with/without
error bars, or you can show the individual triplicate experiments.
Data can be grouped to add additional logic to the plot.

In [ ]:
from omero_screen_plots import cellcycle_plot

for cell_line in df["cell_line"].unique():
    fig, axes = cellcycle_plot(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Display
        title=None,
        cc_phases=True,              # False -> DNA content labels (2N, 4N, etc.)
        show_subG1=False,
        show_significance=True,      # Requires >= 3 plates
        show_repeat_points=True,
        show_plate_legend=False,
        rotation=45,
        colors=None,
        # Figure
        fig_size=(6, 6),
        size_units="cm",
        dpi=300,
        tight_layout=False,
        # Save
        save=False,
        path=None,
        file_format="pdf",
    )

In [ ]:
from omero_screen_plots import cellcycle_stacked

for cell_line in df["cell_line"].unique():
    fig, ax = cellcycle_stacked(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Display
        title=None,
        cc_phases=True,              # False -> DNA content labels (2N, 4N, etc.)
        phase_order=None,            # Custom phase order
        show_triplicates=False,      # True -> individual bars per replicate
        show_error_bars=True,        # Only when show_triplicates=False
        show_boxes=True,             # Only when show_triplicates=True
        show_legend=True,
        colors=None,
        rotation=45,
        x_label=True,
        # Bar settings
        bar_width=0.5,
        repeat_offset=0.18,
        max_repeats=3,
        # Grouping & layout
        group_size=1,
        within_group_spacing=0.2,
        between_group_gap=0.5,
        # Figure
        fig_size=(6, 6),
        size_units="cm",
        dpi=300,
        tight_layout=False,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        # Axes
        axes=None,
    )

## Feature Plots

There are two ways to show features: either as box or violin plots where
individual datapoints for repeats can be overlaid using `feature_plot`,
or with `feature_norm_plot` which uses mode-based normalisation and shows the proportion
above a threshold (default = 1.5x the peak of the distribution).
Set `save_norm_qc=True` for diagnostic data on the normalisation.

In [ ]:
from omero_screen_plots import feature_plot

for cell_line in df["cell_line"].unique():
    fig, ax = feature_plot(
        df=df,
        feature="intensity_mean_EdU_nucleus",
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Display
        violin=False,                # True -> violin plots instead of box plots
        show_scatter=True,
        scale=False,
        ymax=None,                   # float or (min, max) tuple
        title="",
        legend=True,                 # Plate legend
        x_label=True,
        colors=None,
        # Grouping & layout
        group_size=1,
        within_group_spacing=0.2,
        between_group_gap=0.5,
        # Figure
        fig_size=(5, 5),
        size_units="cm",
        dpi=300,
        tight_layout=False,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        # Axes
        axes=None,
    )

In [ ]:
from omero_screen_plots import feature_norm_plot

for cell_line in df["cell_line"].unique():
    fig, ax = feature_norm_plot(
        df=df,
        feature="intensity_mean_EdU_nucleus",
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        # Normalization
        normalize_by_plate=True,
        threshold=1.5,
        save_norm_qc=False,
        # Display
        title="",
        color_scheme="green",       # "green", "blue", or "purple"
        show_triplicates=False,
        show_error_bars=True,
        show_boxes=True,
        legend=False,
        x_label=True,
        # Grouping & layout
        group_size=1,
        within_group_spacing=0.2,
        between_group_gap=0.5,
        # Figure
        fig_size=(8, 6),
        size_units="cm",
        dpi=300,
        tight_layout=False,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        # Axes
        axes=None,
    )

## Classification Plots (optional)

If you ran a classifier during analysis, you can visualise the results here.
First check what classes are available, then generate a list with the order
you want to display them. Classifier columns are named `classifier_{model}` —
e.g. `classifier_nuclei4`, `classifier_mitotic`. Use `df.filter(like="classifier_").columns`
to discover which classifiers are present in your data.

In [ ]:
# Discover classifier columns
classifier_cols = df.filter(like="classifier_").columns.tolist()
print(f"Classifier columns: {classifier_cols}")
# for col in classifier_cols:
#     print(f"{col}: {df[col].unique()}")  # uncomment to check available classes

In [ ]:
from omero_screen_plots import classification_plot

# Set the classifier column to plot (e.g. "classifier_nuclei4")
class_col = classifier_cols[0] if classifier_cols else "classifier_nuclei4"

for cell_line in df["cell_line"].unique():
    fig, ax = classification_plot(
        df=df,
        classes=["normal", "abnormal"],  # Set to your class names
        conditions=conditions,
        condition_col="condition",
        class_col=class_col,
        selector_col="cell_line",
        selector_val=cell_line,
        # Display
        display_mode="stacked",      # or "triplicates"
        title=None,
        y_lim=(0, 100),
        show_legend=True,
        legend_bbox=(0.98, 1.0),
        # Bar settings (stacked mode)
        bar_width=0.75,
        # Triplicates mode settings
        repeat_offset=0.18,
        # Grouping & layout
        group_size=1,
        within_group_spacing=0.2,
        between_group_gap=0.4,
        # Figure
        fig_size=(7, 7),
        size_units="cm",
        dpi=300,
        tight_layout=True,
        # Save
        save=False,
        path=None,
        file_format="pdf",
        # Axes
        axes=None,
    )

## Combining Plots

You can combine individual plots into a multi-panel figure using `matplotlib.pyplot.subplots`.
Pass the `axes` parameter to each plot function to place it in a specific subplot.
This lets you compare different features side by side across conditions and cell lines.

In [ ]:
import matplotlib.pyplot as plt
# from omero_screen_plots import save_fig

# Set the classifier column to plot (e.g. "classifier_nuclei4")
class_col = classifier_cols[0] if classifier_cols else "classifier_nuclei4"

for cell_line in df["cell_line"].unique():
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10, 5))

    classification_plot(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        class_col=class_col,
        classes=["normal", "abnormal"],  # Set to your class names
        display_mode="triplicates",
        group_size=3,
        between_group_gap=0.15,
        within_group_spacing=0.08,
        axes=ax[0, 0],
    )
    ax[0, 0].set_xlabel("")
    ax[0, 0].set_xticklabels([])
    ax[0, 0].set_title("Classification", fontsize=7, weight="bold", x=0.01)

    feature_plot(
        df=df,
        feature="area_cell",
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        violin=True,
        show_scatter=False,
        group_size=3,
        between_group_gap=0.15,
        within_group_spacing=0.08,
        ymax=(0, 15000),
        axes=ax[0, 1],
    )
    ax[0, 1].set_xlabel("")
    ax[0, 1].set_xticklabels([])
    ax[0, 1].set_title("Cell area", fontsize=7, weight="bold", x=0.01)

    cellcycle_stacked(
        df=df,
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        group_size=3,
        between_group_gap=0.15,
        within_group_spacing=0.08,
        show_triplicates=True,
        axes=ax[1, 0],
    )
    ax[1, 0].set_title("Cell cycle", fontsize=7, weight="bold", x=0.01)

    feature_norm_plot(
        df=df,
        feature="intensity_mean_EdU_nucleus",
        conditions=conditions,
        condition_col="condition",
        selector_col="cell_line",
        selector_val=cell_line,
        show_triplicates=True,
        group_size=3,
        between_group_gap=0.15,
        within_group_spacing=0.08,
        threshold=1.5,
        axes=ax[1, 1],
    )
    ax[1, 1].set_title("Feature (normalised)", fontsize=7, weight="bold", x=0.01)

    fig.suptitle(f"{cell_line} Analysis", fontsize=10, weight="bold", y=0.99, x=0.1)
    # save_fig(fig, path_to_data, f"{cell_line}_Analysis")